# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Keywords: {', '.join(metadata.keywords) if hasattr(metadata, 'keywords') else ''}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets and their fields with their @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"Record Set Name: {rs.name}, @id: {rs['@id']}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field['@id']}) | type: {field.data_type}")
    print('-'*40)
# Pick the first record set as example (most datasets have one main table)
main_record_set_id = record_sets[0]['@id'] if record_sets else None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from all record sets into DataFrames
dataframes = {}
for rs in record_sets:
    recs = list(dataset.records(record_set=rs['@id']))
    df = pd.DataFrame(recs)
    dataframes[rs['@id']] = df
    print(f"Loaded record set '{rs.name}' (@id={rs['@id']}) with shape {df.shape}")
    if df.shape[1] > 0:
        print("    Columns (field @id):", list(df.columns))
    print()
# Use the main record set for detailed exploration
main_df = dataframes[main_record_set_id] if main_record_set_id else None
if main_df is not None:
    print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify numeric fields in the main record set (by dtype or field type in metadata)
import numpy as np
numeric_field_id = None
group_field_id = None
if main_df is not None:
    # Attempt to detect first numeric field
    for rs in record_sets:
        if rs['@id'] == main_record_set_id:
            for fld in rs.fields:
                if fld.data_type in ['Integer', 'Float', 'Number']:
                    if fld['@id'] in main_df.columns:
                        numeric_field_id = fld['@id']
                        break
            for fld in rs.fields:
                if fld.data_type == 'Text' and fld['@id'] in main_df.columns:
                    group_field_id = fld['@id']
                    break
    if numeric_field_id and numeric_field_id in main_df.columns:
        print(f"Numeric field for analysis: {numeric_field_id}")
        na_vals = main_df[numeric_field_id].replace(['', None, np.nan], np.nan).dropna().astype(float)
        if not na_vals.empty:
            threshold = float(np.percentile(na_vals, 75))
        else:
            threshold = 10.0
        filtered_df = main_df.copy()
        filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} (75th percentile):")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') -
            pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
        ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
        print(f"\nNormalized column for {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Group by category
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id} (showing means):")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("Main DataFrame is empty or missing.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field_id and numeric_field_id in main_df.columns:
    col_vals = pd.to_numeric(main_df[numeric_field_id], errors='coerce').dropna()
    plt.figure(figsize=(8,4))
    sns.histplot(col_vals, bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id and group_field_id in main_df.columns:
        # Barplot of means by category (top N)
        cat_sizes = main_df[group_field_id].value_counts().sort_values(ascending=False)
        top_cats = cat_sizes.iloc[:10].index.tolist()
        mean_vals = main_df[main_df[group_field_id].isin(top_cats)].groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        plt.figure(figsize=(10,6))
        sns.barplot(y=mean_vals.index, x=mean_vals.values)
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (top 10 categories)")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated metadata access, loading, and exploration of the FAIR² mass spectrometry dataset via the Croissant schema and `mlcroissant`.

Key steps included:
- Programmatic discovery of available record sets and fields using their `@id`.
- Loading tabular data for analysis and basic cleaning.
- Filtering and normalizing numeric variables, grouping and summarizing by category, and visualizing feature distributions.

**Next steps:**
- Explore additional fields or record sets (`@id`) as needed.
- Apply domain-specific data filters or transformations for your scientific analysis.
- Integrate with downstream machine learning pipelines or build custom visualizations based on normalized data.